In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

USER = "root"
PASSWORD = "Roshan123@"
HOST = "localhost"
PORT = 3306
DB = "nyc_collisions"

engine = create_engine(
    f"mysql+pymysql://{USER}:{quote_plus(PASSWORD)}@{HOST}:{PORT}/{DB}"
)

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DATABASE()")).scalar())
    print("Rows in crashes_clean:", conn.execute(text("SELECT COUNT(*) FROM crashes_clean")).scalar())

Connected to: nyc_collisions
Rows in crashes_clean: 0


In [2]:
df = pd.read_csv("collisions_clean.csv", dtype=str, keep_default_na=False, na_values=[""])

# whole-number columns
int_cols = ["collision_id", "year", "month"] + \
           [c for c in df.columns if c.startswith("number_of_")] + ["total_casualties"]
for c in int_cols:
    df[c] = pd.to_numeric(df[c]).astype("int64")

df["hour"] = pd.to_numeric(df["hour"]).astype("Int64")       # Int64 allows blanks
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])
df["crash_date"] = pd.to_datetime(df["crash_date"]).dt.date

print("Shape:", df.shape)
print(df.dtypes)

# longest text in each text column (must fit in the MySQL column size)
text_cols = df.select_dtypes(include="object").columns.drop("crash_date")
print(df[text_cols].apply(lambda s: s.str.len().max()).sort_values(ascending=False).head(10))

Shape: (2269187, 36)
crash_date                        object
crash_time                           str
borough                              str
zip_code                             str
latitude                         float64
longitude                        float64
on_street_name                       str
cross_street_name                    str
off_street_name                      str
number_of_persons_injured          int64
number_of_persons_killed           int64
number_of_pedestrians_injured      int64
number_of_pedestrians_killed       int64
number_of_cyclist_injured          int64
number_of_cyclist_killed           int64
number_of_motorist_injured         int64
number_of_motorist_killed          int64
contributing_factor_vehicle_1        str
contributing_factor_vehicle_2        str
contributing_factor_vehicle_3        str
contributing_factor_vehicle_4        str
contributing_factor_vehicle_5        str
collision_id                       int64
vehicle_type_code_1                 

C:\Users\ROSHAN SHAW\AppData\Local\Temp\ipykernel_5860\3241269885.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns.drop("crash_date")


contributing_factor_vehicle_1    53.0
contributing_factor_vehicle_3    53.0
contributing_factor_vehicle_2    53.0
contributing_factor_vehicle_4    43.0
contributing_factor_vehicle_5    43.0
off_street_name                  40.0
vehicle_type_code_1              38.0
vehicle_type_code_2              38.0
vehicle_type_code_4              35.0
vehicle_type_code_3              35.0
dtype: float64


In [3]:
# 1. length guard: no text may be longer than the MySQL column size
limits = {
    "crash_time": 5, "month_name": 10, "day_of_week": 10, "borough": 20, "zip_code": 10,
    "on_street_name": 100, "cross_street_name": 100, "off_street_name": 100,
    "severity": 25, "vehicle_group_1": 30,
}
for i in range(1, 6):
    limits[f"contributing_factor_vehicle_{i}"] = 100
    limits[f"vehicle_type_code_{i}"] = 100

for col, lim in limits.items():
    longest = df[col].str.len().max()
    assert longest <= lim, f"{col}: longest {longest} > limit {lim}"
print("Length check passed")

# 2. empty the table so re-running never creates duplicates
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE crashes_clean"))

# 3. load in chunks
CHUNK = 100_000
total = len(df)
for start in range(0, total, CHUNK):
    part = df.iloc[start:start + CHUNK]
    part.to_sql("crashes_clean", engine, if_exists="append",
                index=False, method="multi", chunksize=5000)
    print(f"Loaded {min(start + CHUNK, total):,} / {total:,}")

print("Load finished")

Length check passed
Loaded 100,000 / 2,269,187
Loaded 200,000 / 2,269,187
Loaded 300,000 / 2,269,187
Loaded 400,000 / 2,269,187
Loaded 500,000 / 2,269,187
Loaded 600,000 / 2,269,187
Loaded 700,000 / 2,269,187
Loaded 800,000 / 2,269,187
Loaded 900,000 / 2,269,187
Loaded 1,000,000 / 2,269,187
Loaded 1,100,000 / 2,269,187
Loaded 1,200,000 / 2,269,187
Loaded 1,300,000 / 2,269,187
Loaded 1,400,000 / 2,269,187
Loaded 1,500,000 / 2,269,187
Loaded 1,600,000 / 2,269,187
Loaded 1,700,000 / 2,269,187
Loaded 1,800,000 / 2,269,187
Loaded 1,900,000 / 2,269,187
Loaded 2,000,000 / 2,269,187
Loaded 2,100,000 / 2,269,187
Loaded 2,200,000 / 2,269,187
Loaded 2,269,187 / 2,269,187
Load finished


In [4]:
with engine.connect() as conn:
    print("Rows        :", conn.execute(text("SELECT COUNT(*) FROM crashes_clean")).scalar())
    print("Injured     :", conn.execute(text("SELECT SUM(number_of_persons_injured) FROM crashes_clean")).scalar())
    print("Killed      :", conn.execute(text("SELECT SUM(number_of_persons_killed) FROM crashes_clean")).scalar())
    print("Fatal crash :", conn.execute(text("SELECT COUNT(*) FROM crashes_clean WHERE severity='Fatal'")).scalar())

Rows        : 2269187
Injured     : 756353
Killed      : 3617
Fatal crash : 3469
